In [ ]:
from dotenv import load_dotenv
_ = load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
tool = TavilySearchResults(max_results=4) #increased number of results
print(type(tool))
print(tool.name)

Prints:
<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [ ]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)                  # initialize the graph with the state type
        graph.add_node("llm", self.call_openai)         # add a node for calling the LLM, which will return a message that may contain tool calls
        graph.add_node("action", self.take_action)      # add a node for taking actions, which will execute the tool calls and return the results as messages
        graph.add_conditional_edges(                    # add a conditional edge from the LLM node to either the action node or the end state, depending on whether there are tool calls in the LLM's response
            "llm",
            self.exists_action,
            {True: "action", False: END}                # false edge goes to the end state/node, which means the agent is done and can return the final response
        )
        graph.add_edge("action", "llm")                 # add an edge from the action node back to the LLM node, which allows the agent to call the LLM again after executing the tool calls and getting the results
        graph.set_entry_point("llm")
        self.graph = graph.compile()                    
        self.tools = {t.name: t for t in tools}         # create a dictionary of tools for easy lookup by name
        self.model = model.bind_tools(tools)

    # This function checks if there are tool calls in the LLM's response by looking at the last message in the conversation history and checking if it contains any tool calls.
    # If there are tool calls, it returns True, which will trigger the conditional edge to the action node. If there are no tool calls, it returns False, which will trigger the conditional edge to the end state.
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    # This function calls the LLM with the conversation history and returns the response as a message. If there is a system message, it is added to the beginning of the conversation history before calling the LLM.
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    # conditional edge from LLM node goes to this function if there are tool calls in the LLM's response. 
    # This function executes the tool calls and returns the results as messages, which are then added to the conversation history and passed back to the LLM for another round of processing.
    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model="gpt-3.5-turbo")  #reduce inference cost
abot = Agent(model, [tool], system=prompt)

In [ ]:
from IPython.display import Image

# visualize the agent's graph structure
Image(abot.graph.get_graph().draw_png())

Prints: Refer to Image1_lesson2GraphStructure

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]
result = abot.graph.invoke({"messages": messages})

Prints:
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_PvPN1v7bHUxOdyn4J2xJhYOX'}
Back to the model!


In [ ]:
# final state of the agent after running through the graph, which contains the conversation history with the LLM and the results of any tool calls that were made
result

Prints:
{'messages': [HumanMessage(content='What is the weather in sf?'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_PvPN1v7bHUxOdyn4J2xJhYOX', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 153, 'total_tokens': 174, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d1058bf5-2fdf-44af-a004-9468ea6e9a13-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_PvPN1v7bHUxOdyn4J2xJhYOX'}]),
  ToolMessage(content='[{\'url\': \'https://www.easeweather.com/north-america/united-states/california/city-and-county-of-san-francisco/san-francisco/april\', \'content\': \'| Apr. 27 | Sunny Sunny | 16° /9° | 0 mm | 4 |\\n| Apr. 28 | Sunny Sunny | 17° /11° | 0 mm | 4 |\\n| Apr. 29 | Sunny Sunny | 17° /9° | 0 mm | 4 |\\n| Apr. 30 | Overcast Overcast | 16° /9° | 0 mm | 4 |\\n| Next | [...] | Apr. 15 | Sunny Sunny | 13° /9° | 0 mm | 1.6 |\\n| Apr. 16 | Sunny Sunny | 19° /9° | 0 mm | 1.6 |\\n| Apr. 17 | Sunny Sunny | 18° /11° | 0 mm | 1.7 |\\n| Apr. 18 | Sunny Sunny | 18° /11° | 0 mm | 1.8 |\\n| Apr. 19 | Sunny Sunny | 18° /12° | 0 mm | 0 |\\n| Apr. 20 | Sunny Sunny | 19° /12° | 0 mm | 5 |\\n| Apr. 21 | Moderate rain Moderate rain | 16° /12° | 7.5 mm | 3 |\\n| Apr. 22 | Patchy rain possible Patchy rain possible | 15° /13° | 2.6 mm | 3 |\\n| Apr. 23 | Patchy rain possible Patchy rain possible | 15° /12° | 0.5 mm | 3 |\\n| Apr. 24 | Patchy rain possible Patchy rain possible | 14° /11° | 0 mm | 3 |\\n| Apr. 25 | Partly cloudy Partly cloudy | 15° /11° | 0 mm | 4 |\\n| Apr. 26 | Partly cloudy Partly cloudy | 14° /10° | 0 mm | 4 |\\n| Apr. 27 | Sunny Sunny | 16° /9° | 0 mm | 4 | [...] ### Temperatures\\n\\n Until now, April 2026 in San Francisco is slightly cooler than the historical average by -1.5 °F.\\n The forecast for the next days in San Francisco predicts temperatures to be around 16 °F, close to the historical average.\\n In general, the average temperature in San Francisco at the beginning of April is 15 °F. As the month progressed, temperatures tended to moderately rise, reaching an average of 15.6 °F by the end of April.\\n\\n### Rain 🌧️\\n\\n San Francisco experiences moderate rainfall in April, with an average of 6 rainy days and a total precipitation of 18.9 mm.\\n\\n### More\\n\\n New! Chat with our AI weatherman - it’s amazing and free.\\n\\nAverage dry days\\n\\n24\\n\\nDry days\\n\\nAverage rainy days\\n\\n6\\n\\nRainy days\\n\\nAverage snowy days\\n\\n0\\n\\nSnow days\\n\\nAverage temperatures\\n\\n15°/9°\'}, {\'url\': \'https://en.climate-data.org/north-america/united-states-of-america/california/san-francisco-385/t/april-4/\', \'content\': "| 23. April | 13 °C | 56 °F | 18 °C | 65 °F | 10 °C | 50 °F | 11 °C | 52 °F | 0.7 mm | 0.0 inch. |\\n| 24. April | 13 °C | 56 °F | 18 °C | 64 °F | 10 °C | 50 °F | 11 °C | 52 °F | 0.9 mm | 0.0 inch. |\\n| 25. April | 13 °C | 56 °F | 18 °C | 64 °F | 10 °C | 50 °F | 11 °C | 52 °F | 1.1 mm | 0.0 inch. |\\n| 26. April | 13 °C | 56 °F | 18 °C | 65 °F | 9 °C | 49 °F | 11 °C | 52 °F | 0.5 mm | 0.0 inch. |\\n| 27. April | 14 °C | 56 °F | 19 °C | 66 °F | 10 °C | 49 °F | 11 °C | 52 °F | 0.3 mm | 0.0 inch. |\\n| 28. April | 14 °C | 57 °F | 19 °C | 66 °F | 10 °C | 50 °F | 11 °C | 52 °F | 1.0 mm | 0.0 inch. |\\n| 29. April | 14 °C | 57 °F | 20 °C | 67 °F | 10 °C | 50 °F | 11 °C | 52 °F | 0.7 mm | 0.0 inch. |\\n| 30. April | 14 °C | 58 °F | 20 °C | 68 °F | 10 °C | 50 °F | 11 °C | 52 °F | 0.1 mm | 0.0 inch. | [...] ## \\n\\n## \\n\\n# San Francisco Weather in April\\n\\nAre you planning a holiday with hopefully nice weather in San Francisco in April 2026? Here you can find all information about the weather in San Francisco in April:\\n\\n## San Francisco weather in April\\n\\n|  |  |  |  |  |  |\\n ---  ---  --- |\\n|  | Temperature April | 12.5°C | 54.6°F |  | Precipitation / Rainfall April | 40mm | 1.6 inches |\\n|  | Temperature April max. | 17.4°C | 63.3°F |  | Water Temperature April | 11°C | 52°F |\\n|  | Temperature April min. | 8.9°C | 48.1°F |\\n\\n|  |  |  |\\n --- \\n|  | Temperature April | 12.5°C | 54.6°F |\\n|  | Temperature April max. | 17.4°C | 63.3°F ||  | Temperature April min. | 8.9°C | 48.1°F |\\n|  | Precipitation / Rainfall April | 40mm | 1.6 inches |\\n|  | Water Temperature April | 11°C | 52°F | [...] April in San Francisco offers a mild and comfortable climate, ideal for exploring the city\'s numerous attractions. The average temperature during this month is around 54.5°F (12.5°C), providing a pleasant atmosphere for both locals and tourists. Daytime highs typically reach about 63.3°F (17.4°C), perfect for outdoor activities like walking across the Golden Gate Bridge or visiting Alcatraz Island. Meanwhile, nighttime lows generally dip to around 48°F (8.9°C), so it\'s wise to carry a light jacket during evening outings. This temperate weather allows visitors to enjoy various festivals and events that take place in April without the discomfort of extreme temperatures, making it one of the best times to experience all that San Francisco has to offer."}, {\'url\': \'https://polymarket.com/event/highest-temperature-in-san-francisco-on-april-27-2026\', \'content\': \'Trader consensus on Polymarket heavily favors a high temperature of 62-63°F (42.5% implied probability) for San Francisco on April 27, driven by the latest National Weather Service guidance and high-resolution forecast models like HRRR and NAM, which project daytime maxima near 62°F at San Francisco International Airport (KSFO) amid persistent marine stratus. A low-pressure system offshore has strengthened onshore flow and deepened the coastal stratus deck, limiting solar heating and diurnal temperature rises typical of spring—where April 27 climatological normals average 64.6°F. Recent model runs over the past 24 hours show minimal boundary-layer mixing and low clearing potential (25% precipitation odds), suppressing highs below 65°F; however, unexpected afternoon burn-off could nudge [...] Trader consensus on Polymarket heavily favors a high temperature of 62-63°F (42.5% implied probability) for San Francisco on April 27, driven by the latest National Weather Service guidance and high-resolution forecast models like HRRR and NAM, which project daytime maxima near 62°F at San Francisco International Airport (KSFO) amid persistent marine stratus. A low-pressure system offshore has strengthened onshore flow and deepened the coastal stratus deck, limiting solar heating and diurnal temperature rises typical of spring—where April 27 climatological normals average 64.6°F. Recent model runs over the past 24 hours show minimal boundary-layer mixing and low clearing potential (25% precipitation odds), suppressing highs below 65°F; however, unexpected afternoon burn-off could nudge [...] $802 Vol.\\n\\n33%\\n\\n62-63°F\\n\\n$1,210 Vol.\\n\\n44%\\n\\n64-65°F\\n\\n$1,334 Vol.\\n\\n16%\\n\\n66-67°F\\n\\n$3,592 Vol.\\n\\n2%\\n\\n68°F or higher\\n\\n$549 Vol.\\n\\n1%\\n\\n62-63°F 44%\\n\\n60-61°F 33%\\n\\n64-65°F 16%\\n\\n58-59°F 8%\\n\\n$13,099 Vol.\\n\\n$13,099 Vol.\\n\\n49°F or below\\n\\n<1%\\n\\n50-51°F\\n\\n<1%\\n\\n52-53°F\\n\\n<1%\\n\\n54-55°F\\n\\n1%\\n\\n56-57°F\\n\\n1%\\n\\n58-59°F\\n\\n8%\\n\\n60-61°F\\n\\n33%\\n\\n62-63°F\\n\\n44%\\n\\n64-65°F\\n\\n16%\\n\\n66-67°F\\n\\n2%\\n\\n68°F or higher\\n\\n1%\\n\\n## Rules\\n\\n## Market Context\\n\\nMarket Opened: Apr 25, 2026, 12:20 AM ET\\n\\nResolution Source\\n\\nResolver\'}, {\'url\': \'https://www.weather25.com/north-america/usa/california/san-francisco?page=month&month=April\', \'content\': \'Click on a day for an hourly weather forecast\\n\\nThursday Apr 23 Image 8: Sunny 0 mm 20°/9°Friday Apr 24 Image 9: Cloudy 0 mm 15°/10°Saturday Apr 25 Image 10: Patchy rain possible 0 mm 14°/12°Sunday Apr 26 Image 11: Patchy rain possible 0.3 mm 15°/12°Monday Apr 27 Image 12: Overcast 0 mm 16°/11°Tuesday Apr 28 Image 13: Partly cloudy 0 mm 16°/12°Wednesday Apr 29 Image 14: Sunny 0 mm 15°/12°Thursday Apr 30 Image 15: Sunny 0 mm 14°/11°Friday May 1 Image 16: Sunny 0 mm 15°/11°Saturday May 2 Image 17: Overcast 0 mm 15°/11°Sunday May 3 Image 18: Patchy rain possible 0 mm 15°/11°Monday May 4 Image 19: Sunny 0 mm 15°/12°Tuesday May 5 Image 20: Sunny 0 mm 17°/12°Wednesday May 6 Image 21: Cloudy 0 mm 17°/10°Next Month >>\\n\\n## The average weather in San Francisco in April [...] ## The average weather in San Francisco in April\\n\\nThe temperatures in San Francisco in April are quite cold with temperatures between 9°C and 18°C, warm clothes are a must.\\n\\nThe wather in San Francisco in April can vary between cold and nice weather days. Expect a few rainy days but usually not more than 3.\\n\\nOur weather forecast can give you a great sense of what weather to expect in San Francisco in April 2026.\\n\\nIf you’re planning to visit San Francisco in the near future, we highly recommend that you review the 14 day weather forecast for San Francisco before you arrive.\\n\\nImage 22: Temperatures\\n\\nTemperatures\\n\\n18° /9° \\n\\nImage 23: Rainy Days\\n\\nRainy Days\\n\\n2\\n\\nImage 24: Snowy Days\\n\\nSnowy Days\\n\\n0\\n\\nImage 25: Dry Days\\n\\nDry Days\\n\\n28\\n\\nImage 26: Rainfall\\n\\nRainfall\\n\\n22\\n\\nmm\\n\\nImage 27: 11.0 [...] | 19 Image 46: Partly cloudy 15°/10° | 20 Image 47: Partly cloudy 15°/9° | 21 Image 48: Partly cloudy 17°/10° | 22 Image 49: Partly cloudy 16°/10° | 23 Image 50: Sunny 20°/9° | 24 Image 51: Cloudy 15°/10° | 25 Image 52: Patchy rain possible 14°/12° |\\n| 26 Image 53: Patchy rain possible 15°/12° | 27 Image 54: Overcast 16°/11° | 28 Image 55: Partly cloudy 16°/12° | 29 Image 56: Sunny 15°/12° | 30 Image 57: Sunny 14°/11° |  |  |\'}]', name='tavily_search_results_json', tool_call_id='call_PvPN1v7bHUxOdyn4J2xJhYOX'),
  AIMessage(content='The weather in San Francisco on April 27, 2026, is expected to be sunny with a temperature of 16°C (61°F) during the day and 9°C (48°F) at night. There is no precipitation expected on that day.', response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 3257, 'total_tokens': 3311, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-6721f04a-5525-4caf-8e02-c1050ab1b6b2-0')]}

In [ ]:
# actual content of the final message from the LLM, which should contain the answer to the user's question after any necessary tool calls were made and their results were processed
result['messages'][-1].content

Prints:
'The weather in San Francisco on April 27, 2026, is expected to be sunny with a temperature of 16°C (61°F) during the day and 9°C (48°F) at night. There is no precipitation expected on that day.'

In [ ]:
messages = [HumanMessage(content="What is the weather in SF and LA?")]
result = abot.graph.invoke({"messages": messages})

Prints:

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_3AAaDSfchif60Drx9QyyOiZS'}

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_Ldw8ZaATItg9oW0SRBlhKtpc'}
Back to the model!


In [ ]:
result['messages'][-1].content

Prints:

'The weather in San Francisco for April 27, 2026, is expected to be sunny with a high of 16°C and a low of 11°C. The weather in Los Angeles for the same date is forecasted to be partly cloudy with a high of 19°C and a low of 14°C.'

In [ ]:
# Note, the query was modified to produce more consistent results. 
# Results may vary per run and over time as search information and models change.

query = "Who won the super bowl in 2024? In what state is the winning team headquarters located? \
What is the GDP of that state? Answer each question." 
messages = [HumanMessage(content=query)]

model = ChatOpenAI(model="gpt-4o")  # requires more advanced model
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})

Prints:

Calling: {'name': 'tavily_search_results_json', 'args': {'query': '2024 Super Bowl winner'}, 'id': 'call_HBUU1Lo9WSgKCPKYCAStSb7g'}
Back to the model!

Calling: {'name': 'tavily_search_results_json', 'args': {'query': '2024 Super Bowl winner'}, 'id': 'call_OW0B9IkbdcmdZuaMwzu1WtwR'}
Back to the model!


In [ ]:
print(result['messages'][-1].content)